# Updating data from GenBank

Author: Alexander Maksiaev

Purpose: Update labels from previously gotten data from GISAID + Andersen, using Genbank.

In [1]:
# Housekeeping


import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import dateutil 
from datetime import datetime
from collections import defaultdict 
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# Make sure you have the correct paths

date_range = "01-01-2024--04-14-2025"
update_date = "05-21-2025"

home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
originals = downloads + "GISAID_Andersen_Combined_Files/"
temp_files = downloads + "Andersen_Temp_Files/"
github_files = downloads + "Andersen_Downloads/avian-influenza/metadata/"
complete = originals + date_range + "_B3_13_D1_1/D1_1/"
# home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
# originals = downloads + "Combinations/GISAID_Andersen/B3_13_D1_1/" 
# temp_files = downloads + "Andersen/"



os.chdir(complete)

## Collection Dates

In [2]:
# Upload saved data 
# os.chdir(temp_files + "saved/")
os.chdir(github_files)
metadata = pd.read_csv("SraRunTable_automated.csv") 
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
metadata_genbank = metadata.merge(genbank_mapping, on="Run")
# metadata_genbank = pd.read_csv("metadata_genbank_4-18-2025.csv") # Since 1/1/2024
os.chdir(originals)

display(metadata_genbank)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,BioSample Accession,is_retracted,retraction_detection_date_utc,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name
0,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,SRS21079812,False,NaN,SRR28752446_HA_cns.fa,Consensus_SRR28752446_HA_cns_threshold_0.5_qua...,SRR28752446,HA,PP740722.1,4,A/blackbird/Texas/24-008354-001/2024
1,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,SRS21079812,False,NaN,SRR28752446_MP_cns.fa,Consensus_SRR28752446_MP_cns_threshold_0.5_qua...,SRR28752446,MP,PP740723.1,7,A/blackbird/Texas/24-008354-001/2024
2,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,SRS21079812,False,NaN,SRR28752446_NA_cns.fa,Consensus_SRR28752446_NA_cns_threshold_0.5_qua...,SRR28752446,NaN,PP740724.1,6,A/blackbird/Texas/24-008354-001/2024
3,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,SRS21079812,False,NaN,SRR28752446_NP_cns.fa,Consensus_SRR28752446_NP_cns_threshold_0.5_qua...,SRR28752446,NP,PP740725.1,5,A/blackbird/Texas/24-008354-001/2024
4,SRR28752446,WGS,146.11,93605195,PRJNA1102327,SAMN41019184,Viral,30074178,USDA-NVSL,2024,...,SRS21079812,False,NaN,SRR28752446_NS_cns.fa,Consensus_SRR28752446_NS_cns_threshold_0.5_qua...,SRR28752446,NS,PP740726.1,8,A/blackbird/Texas/24-008354-001/2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38713,SRR32633094,WGS,148.09,109655942,PRJNA1102327,SAMN47290845,Viral,41506380,USDA-NVSL,2025,...,SRS24304292,False,NaN,SRR32633094_NP_cns.fa,Consensus_SRR32633094_NP_cns_threshold_0.5_qua...,SRR32633094,NP,PV457233.1,5,A/cattle/CA/25-005648-001-original/2025
38714,SRR32633094,WGS,148.09,109655942,PRJNA1102327,SAMN47290845,Viral,41506380,USDA-NVSL,2025,...,SRS24304292,False,NaN,SRR32633094_NS_cns.fa,Consensus_SRR32633094_NS_cns_threshold_0.5_qua...,SRR32633094,NS,PV457236.1,8,A/cattle/CA/25-005648-001-original/2025
38715,SRR32633094,WGS,148.09,109655942,PRJNA1102327,SAMN47290845,Viral,41506380,USDA-NVSL,2025,...,SRS24304292,False,NaN,SRR32633094_PA_cns.fa,Consensus_SRR32633094_PA_cns_threshold_0.5_qua...,SRR32633094,PA,PV457231.1,3,A/cattle/CA/25-005648-001-original/2025
38716,SRR32633094,WGS,148.09,109655942,PRJNA1102327,SAMN47290845,Viral,41506380,USDA-NVSL,2025,...,SRS24304292,False,NaN,SRR32633094_PB1_cns.fa,Consensus_SRR32633094_PB1_cns_threshold_0.5_qu...,SRR32633094,PB1,PV457230.1,2,A/cattle/CA/25-005648-001-original/2025


In [3]:
# no_updates = pd.DataFrame()
# no_updates_isolate = []

# Get only labels that have no states or collection dates, and update them

def update(file_name, update_date):
    updates = {}
    with open(file_name) as f:
        lines = f.readlines()
        for i, line in enumerate(lines):
            if line[0] == ">": # It's a header
                header = line 
                collection_date = header.split("|")[-3]
                state = header.split("/")[2]
                state = state.replace(": ", "-")
                # geo_location_collection_date = state + "|" + collection_date
                geo_location_collection_date = collection_date
                isolate = header.split("/")[3]
                sequence = lines[i + 1] # Sequence always comes in one line after header
                if "-" not in collection_date: # If there are no dashes, i.e. if it's just the year
                    # Find the correct collection date, if it exists
                    row = metadata_genbank[metadata_genbank["isolate"] == isolate]
                
                    try: # Isolate may not be in this dataset
                        id = row["BioSample"].values[0]
                        print(id)
                        geo_location_collection_date = search_collection_date(id, row) # Update unknown dates, if possible
                    except:
                        print("No BioSample nor date found for isolate", isolate)
                            # no_updates_isolate.append(isolate)

                if state == "USA": # If we don't have a state
                    row = metadata_genbank[metadata_genbank["isolate"] == isolate]

                    try: # Isolate may not be in this dataset
                        genbank_name = row["genbank_name"].values[0]
                        state = genbank_name.split("/")[2]
                        print(state)
                    except:
                        print("No state found for isolate", isolate)
                        # no_updates_isolate.append(isolate)

                updates[header] = [state, geo_location_collection_date, sequence]

        f.close()

    updates_df = pd.DataFrame.from_dict(updates, orient="index", columns=["state", "geo_location_collection_date", "sequence"])
    updates_df["header"] = updates_df.index
    updates_df = updates_df.reset_index()

    updated_file_name = ".".join(file_name.split(".")[:-1]) + "_" + date_range + "_" + update_date + "_update." + file_name.split(".")[-1]

    with open(updated_file_name, "w") as g:

        for i, row in updates_df.iterrows():
            header = row["header"]
            # print(header)
            # print(header.split("|")[-3])
            
            if header.split("/")[2] == "USA":
                header = header.replace(header.split("/")[2], row["state"])
            
            # header = header.replace(str(header.split("|")[-3]), str(row["geo_location_collection_date"])) # Only the first instance is replaced
            header = str(row["geo_location_collection_date"]).join(header.rsplit(str(header.split("|")[-3]), 1))
            g.write(header)
            g.write(row["sequence"])

        g.close()

    # no_updates["isolate"] = no_updates_isolate
    # no_updates.to_csv("not_updated.csv")


In [ ]:
# Create files with updates

# os.chdir(originals)

# for dirpath, dirs, files in os.walk(originals + date_range + "_B3_13_D1_1/"): # Find the fasta file
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         update(file_name, update_date)
#     break 

os.chdir(complete)
# file = "8-newid_B3.13_APR14_NS_trim_codon_aln_n3471_FINAL.fasta"
# update(file, update_date)

for dirpath, dirs, files in os.walk(complete): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file)
        update(file_name, update_date)
    break 

No BioSample nor date found for isolate 25-002460-001
No state found for isolate 25-002460-001
No BioSample nor date found for isolate 25-002634-002
No state found for isolate 25-002634-002
No BioSample nor date found for isolate 002453-003
No state found for isolate 002453-003
No BioSample nor date found for isolate 005883-002
No state found for isolate 005883-002
No BioSample nor date found for isolate 003547-001
No state found for isolate 003547-001
No BioSample nor date found for isolate 004498-005
No state found for isolate 004498-005
No BioSample nor date found for isolate 007118-006
No state found for isolate 007118-006
No BioSample nor date found for isolate 007118-086
No state found for isolate 007118-086
No BioSample nor date found for isolate 009029-001
No state found for isolate 009029-001
No BioSample nor date found for isolate 003926-003
No state found for isolate 003926-003
No BioSample nor date found for isolate 25-006702-001
No state found for isolate 25-006702-001
No 

In [ ]:
# search_collection_date_term("PP752829.1", metadata_genbank)